## 1. Import libraries

In [2]:
import torch
import torch.nn as nn
import torch.quantization as quant
import numpy as np
import joblib
import os
from sklearn.metrics import accuracy_score, classification_report

## 2. Define CNN-LSTM Architecture

In [ ]:
class CNNLSTM(nn.Module):
    def __init__(self, num_classes=2):
        super(CNNLSTM, self).__init__()
        
        # CNN layers for feature extraction from spectrograms
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout_cnn = nn.Dropout(0.3)
        
        # LSTM layer
        self.lstm = nn.LSTM(input_size=256 * 8 * 8, hidden_size=128, batch_first=True, dropout=0.3)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, num_classes)
        self.dropout_fc = nn.Dropout(0.3)
    
    def forward(self, x):
        # CNN forward pass
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        
        # Flatten for LSTM
        x = x.view(x.size(0), -1)
        
        # Reshape to (batch, seq_len=1, features)
        x = x.unsqueeze(1)
        
        # LSTM forward pass
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        
        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        
        return x